In [ ]:
// #r "./binaries/BoSSSpad.dll"
// #r "./binaries/LsTest.dll"
#r "BoSSSpad.dll"
#r "LsTest.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Platform.LinAlg;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Solution.LevelSetTools;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Application.LsTest;
Init();

# Load Sessions

Init Workflowmanagement and Project

In [ ]:
BoSSSshell.WorkflowMgm.Init("SwirlingFlow_TemporalConvergence");
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination);

In [ ]:
sessions

In [ ]:
using System.IO;

In [ ]:
// var s = sessions.Pick(0);
// s.Export().To(Path.GetFullPath("./Plots/"+s.Name)).WithSupersampling(2).Do();

In [ ]:
var tab = sessions.GetSessionTable();

In [ ]:
var Evos = tab.GetColumn<string>("id:Evo").Distinct();
Evos.ForEach(e => Console.WriteLine(e));

var hRes = tab.GetColumn<int>("id:Res").Distinct();
hRes.ForEach(e => Console.WriteLine(e));

var pRes = tab.GetColumn<int>("id:Degree").Distinct();
pRes.ForEach(e => Console.WriteLine(e));

In [ ]:
List<Plot2Ddata> plts = new List<Plot2Ddata>();

foreach(int h in hRes){
    foreach(string Evo in Evos){
        var plt = new Plot2Ddata().WithLogX().WithLogY();
        foreach(int p in pRes){
            // select and sort
            var sess = sessions.Where(s => Convert.ToString(s.KeysAndQueries["id:Evo"]) == Evo &&  Convert.ToInt32(s.KeysAndQueries["id:Res"]) == h && Convert.ToInt32(s.KeysAndQueries["id:Degree"]) == p).ToArray();
            Array.Sort(sess, (a,b) => Convert.ToDouble(a.KeysAndQueries["id:dt"]).CompareTo(Convert.ToDouble(b.KeysAndQueries["id:dt"])));
            sess.ForEach(s => Console.WriteLine(s.Name));

            var Ref = new BoSSS.Application.LsTest.LevelSetUnitTests.LevelSetSwirlingFlowTest(2, p, false, 0.0); // to get the "Phi" Functions etc.

            Func<ITimestepInfo, double> errorFunctional = ts => {
                var PhiDG = ts.Fields.Single(f => f.Identification == "PhiDG");
                var PhiCG = ts.Fields.Single(f => f.Identification == "Phi");

                // use the same domain for all error evaluations
                var RefLS = new LevelSet(PhiCG.Basis, "PhiRef");
                RefLS.ProjectField(1.0, Ref.GetPhi()[0].Vectorize().SetTime(0.0));
                LevelSetTracker LsTrk = new LevelSetTracker((GridData)PhiCG.GridDat, XQuadFactoryHelper.MomentFittingVariants.Saye, 1, new string[] { "A", "B" }, RefLS);
                LsTrk.UpdateTracker(0.0);

                double err = 0.0;
                err = PhiDG.L2Error(Ref.GetPhi()[0].Vectorize().SetTime(0.0), 16, new BoSSS.Foundation.Quadrature.CellQuadratureScheme(true, LsTrk.Regions.GetNearFieldMask(1)));
                return err;
            };

            var errs = sess.Select(s => errorFunctional(s.Timesteps.Last())).ToArray();
            errs.ForEach(err => Console.WriteLine(err));
            var _plt = sess.ToTimeConvergenceData(errorFunctional);
            _plt.dataGroups.Single().Name = " Degree: " + p;
            plt = plt.Merge(_plt);
        }        
        plts.Add(plt);
        plt.Title = "Res: " + h + " Evo: " + Evo;
        plt.TitleFont = 30;
        plt.tmargin = 8;
        plt.ModFormat();
    }
}

In [ ]:
foreach(var plt in plts){
    display(plt.Title + " : ");
    plt.Regression().ForEach( g => display( g.Key + ", Slope : " + g.Value));
    display(plt.PlotNow());
    display(":::::::::::::::::::::::::::::::::::::::::");
}